# httpbin Deployment 弹性伸缩

本 Notebook 独立创建 httpbin Tool 和 Deployment，先使用 `MinInstanceCount=0` 观察按需启动，再更新为两个常驻实例，并同时调整实例上限与单实例请求并发租约。它展示配置与可观察请求，不是压力测试。

> ID 由读者从输出中手工复制；不使用提取或轮询脚本。请把 `AGR_ROLE_ARN` 替换为允许 AGR 拉取目标 CCR 镜像的 CAM 角色 ARN。

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-scaling-your-name
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-scaling-your-name
!agr status

## 1. 创建独立 Tool

修改两个名称中的 `your-name` 后创建 Tool。

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. 从零实例开始

复制 `ToolId`。初始配置允许没有活跃实例，最多扩到三个实例，每个实例一次只持有一个请求或连接 Lease。首次请求将触发按需启动。

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID" \
  --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":3,"MaxInstanceRequestConcurrency":1}' \
  --lifecycle-configuration '{"IdleTimeoutSeconds":60,"IdleAction":"STOP"}'

## 3. 触发按需启动

复制 `DeploymentId`，获取短期 Token，再复制 `Data.Response.Response.Token`。第一次访问可能包含实例启动延迟；后续访问通常复用已启动容量。

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error \
  --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" \
  "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"

## 4. 切换为常驻容量

`deployment update` 对伸缩对象执行完整替换，因此必须同时提供三个字段。下面把活跃实例下限改为 `2`，上限改为 `4`，并允许每个实例同时持有 `10` 个请求或连接 Lease。更新后用 `get` 确认配置；实例数会由服务异步收敛。

In [ ]:
!agr deployment update "$HTTPBIN_DEPLOYMENT_ID" \
  --region "$AGR_REGION" \
  --scaling-configuration '{"MinInstanceCount":2,"MaxInstanceCount":4,"MaxInstanceRequestConcurrency":10}'
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"

## 参数含义

- `MinInstanceCount` 是活跃实例下限；设为 `0` 可按需缩至零，设为正数可保持常驻容量。
- `MaxInstanceCount` 是活跃实例上限，且不能小于下限。
- `MaxInstanceRequestConcurrency` 限制每个活跃实例同时持有的 Deployment 请求或连接 Lease；它不是整个 Deployment 的全局并发上限。

## 5. 清理资源

即使观察结果与预期不同，也先保留命令输出用于排查，再执行清理。

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

若仍有非 `STOPPED` 实例，复制实例 ID，并在新单元先执行 `%env HTTPBIN_INSTANCE_ID=replace-me`，再执行 `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`；逐个处理后再删除 Tool。

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait